Task 1: Implement quantization from scratch (30 min)


In [1]:
import numpy as np

def quantize_int8(x):

    x = np.asarray(x, dtype=np.float32)
    qmin, qmax = -128, 127

    x_min, x_max = float(x.min()), float(x.max())
    if x_max == x_min: # guard against a constant array
        x_max = x_min + 1e-8 # the no. denominator could not be zero

    scale = (x_max - x_min) / (qmax - qmin)
    zero_point = qmin - x_min / scale
    zero_point = int(round(zero_point))
    zero_point = max(qmin, min(qmax, zero_point))

    q = np.round(x / scale + zero_point)
    q = np.clip(q, qmin, qmax).astype(np.int8)

    return q, scale, zero_point

# Reverse the Quantization
def dequantize_int8(q, scale, zero_point):
    q = np.asarray(q, dtype=np.int32)
    return (q - zero_point) * scale

In [2]:
# Test on a small array
sample = np.array([-2.5, -1.0, 0.0, 0.3, 1.8, 3.2, 5.0], dtype=np.float32)

q, scale, zero_point = quantize_int8(sample)
recovered = dequantize_int8(q, scale, zero_point)

print("Original:    ", np.round(sample, 4))
print("Quantized:   ", q)
print("Scale:       ", round(scale, 6))
print("Zero point:  ", zero_point)
print("Dequantized: ", np.round(recovered, 4))
print("Abs error:   ", np.round(np.abs(sample - recovered), 4))
print("Max abs error:", round(float(np.max(np.abs(sample - recovered))), 4))

Original:     [-2.5 -1.   0.   0.3  1.8  3.2  5. ]
Quantized:    [-128  -77  -43  -33   18   66  127]
Scale:        0.029412
Zero point:   -43
Dequantized:  [-2.5    -1.      0.      0.2941  1.7941  3.2059  5.    ]
Abs error:    [0.     0.     0.     0.0059 0.0059 0.0059 0.    ]
Max abs error: 0.0059


In [4]:
# Quantization is the process of converting large model or value into minimum value that we can convert for maximize the memory efficiency to save memory.
# In this code we are converting the large floating point calue into small integer. like we don't need precison all the time so whenever the approximation needed then we can quantized the value or the specific model.

Task 2: Estimate model size from parameter count (30 min)

In [5]:
def model_size(num_params, dtype="fp32"):
    """
    Estimate a model's on-disk / in-memory size given its parameter count and dtype.
    Returns (size_in_MB, size_in_GB).
    """
    bytes_per_param = {
        "fp32": 4,
        "fp16": 2,
        "bf16": 2,
        "int8": 1,
    }
    if dtype not in bytes_per_param:
        raise ValueError(f"Unsupported dtype: {dtype!r}. Choose from {list(bytes_per_param)}")

    total_bytes = num_params * bytes_per_param[dtype]
    size_mb = total_bytes / (1024 ** 2)
    size_gb = total_bytes / (1024 ** 3)
    return size_mb, size_gb

In [6]:
example_models = [
    ("125M model", 125_000_000),
    ("1B model",   1_000_000_000),
    ("7B model",   7_000_000_000),
]

print(f"{'Model':<12}{'dtype':<8}{'Size (MB)':>14}{'Size (GB)':>12}")
print("-" * 46)
for name, params in example_models:
    for dtype in ["fp32", "fp16", "int8"]:
        mb, gb = model_size(params, dtype)
        print(f"{name:<12}{dtype:<8}{mb:>14,.1f}{gb:>12.3f}")
    print()

Model       dtype        Size (MB)   Size (GB)
----------------------------------------------
125M model  fp32             476.8       0.466
125M model  fp16             238.4       0.233
125M model  int8             119.2       0.116

1B model    fp32           3,814.7       3.725
1B model    fp16           1,907.3       1.863
1B model    int8             953.7       0.931

7B model    fp32          26,702.9      26.077
7B model    fp16          13,351.4      13.039
7B model    int8           6,675.7       6.519



In [ ]:
# In the Task we calcuating the size of the model on the basis of floating points and with bytes_per_params.
# After calculating the model is of how many bytes then immediately we divid it to get the bytes size in MB and GB.

## Task 3: System design scenario — multi-GPU inference (30 min)

Scenario: You need to serve a large language model that doesn't fit on a single GPU. You have access to a node with 8
GPUs connected via NVLink, and multiple such nodes connected via InfiniBand.


### 1. Where would you split the model, and why?

The model should be split in **two levels**:

* **Within the same node (8 GPUs):** Use **Tensor Parallelism (TP)**. The model's calculations are divided among the 8 GPUs. Since these GPUs need to communicate frequently, **NVLink** is ideal.
* **Across different nodes:** Use **Pipeline Parallelism (PP)**. Different groups of model layers are placed on different nodes. For example, Node 1 can handle layers 1–20 and Node 2 can handle layers 21–40.

This arrangement matches the communication needs of the model with the available hardware.

### 2. What role does NVLink play vs InfiniBand?

**NVLink** connects GPUs **inside the same node**. It is extremely fast and has low latency, so it is suitable for the frequent communication required by **Tensor Parallelism**.

**InfiniBand** connects **different servers/nodes**. It is also very fast, but slower than NVLink, so it is better suited for communication between pipeline stages.

**Easy memory trick:**

**NVLink → GPU ↔ GPU → Inside node**
**InfiniBand → Node ↔ Node → Between nodes**

### 3. What would go wrong if their roles were swapped?

If we use **Tensor Parallelism across nodes using InfiniBand**, GPUs would have to communicate over the network very frequently. This increases communication delay, causing GPUs to wait for each other and making inference much slower.

If we use **Pipeline Parallelism inside a node**, the system will still work, but we would not be using the high-speed NVLink connection efficiently.

So, swapping them can reduce overall performance.

### 4. Short Design Note

**Design:** Use **Tensor Parallelism across the 8 GPUs within each node using NVLink**, because TP requires frequent and fast GPU-to-GPU communication. Across nodes, use **Pipeline Parallelism over InfiniBand**, where the model is divided into groups of layers and each node processes a different stage. NVLink provides very fast, low-latency communication inside a node, while InfiniBand provides high-speed communication between nodes. Using InfiniBand for tensor parallelism would create a communication bottleneck because TP requires frequent synchronization. Therefore, the best design is to keep **fine-grained communication on NVLink** and use **InfiniBand for coarser communication between nodes**.


Task 4: Compute-bound vs memory-bound estimation (30 min)


In [ ]:
def bound_analysis(flops_per_token, bytes_per_token, compute_flops_per_sec, mem_bandwidth_bytes_per_sec):

    time_compute = flops_per_token / compute_flops_per_sec
    time_memory = bytes_per_token / mem_bandwidth_bytes_per_sec

    bound = "compute-bound" if time_compute >= time_memory else "memory-bound"

    return {
        "bound": bound,
        "time_compute_s": time_compute,
        "time_memory_s": time_memory,
        "arithmetic_intensity_flops_per_byte": flops_per_token / bytes_per_token,
        "hardware_ridge_point_flops_per_byte": compute_flops_per_sec / mem_bandwidth_bytes_per_sec,
    }

In [9]:
# GPU profile similar to an A100: ~312 TFLOP/s (bf16 tensor cores), ~2 TB/s memory bandwidth
COMPUTE = 312e12       # FLOPs/sec
BANDWIDTH = 2e12       # bytes/sec

# A 7B-parameter model in fp16 -> ~14e9 bytes of weights, ~2 FLOPs per parameter per token (standard estimate)
params = 7e9
flops_per_token_single = 2 * params           # forward pass, single token
bytes_per_token_single = params * 2           # fp16 weights streamed once per token, batch = 1

configs = [
    {
        "name": "Decode, batch=1 (7B, fp16)",
        "flops_per_token": flops_per_token_single,
        "bytes_per_token": bytes_per_token_single,
    },
    {
        "name": "Prefill/decode, batch=64 (7B, fp16, weights reused across batch)",
        "flops_per_token": flops_per_token_single * 64,
        "bytes_per_token": bytes_per_token_single,        # same weights loaded once, shared by the whole batch
    },
    {
        "name": "Prefill/decode, batch=512 (7B, fp16, weights reused across batch)",
        "flops_per_token": flops_per_token_single * 512,
        "bytes_per_token": bytes_per_token_single,
    },
]

for cfg in configs:
    result = bound_analysis(cfg["flops_per_token"], cfg["bytes_per_token"], COMPUTE, BANDWIDTH)
    print(cfg["name"])
    print(f"  -> {result['bound']}")
    print(f"     time_compute = {result['time_compute_s']*1e3:.4f} ms   "
          f"time_memory = {result['time_memory_s']*1e3:.4f} ms")
    print(f"     arithmetic intensity = {result['arithmetic_intensity_flops_per_byte']:.2f} FLOPs/byte  "
          f"(hardware ridge point = {result['hardware_ridge_point_flops_per_byte']:.2f} FLOPs/byte)")
    print()

Decode, batch=1 (7B, fp16)
  -> memory-bound
     time_compute = 0.0449 ms   time_memory = 7.0000 ms
     arithmetic intensity = 1.00 FLOPs/byte  (hardware ridge point = 156.00 FLOPs/byte)

Prefill/decode, batch=64 (7B, fp16, weights reused across batch)
  -> memory-bound
     time_compute = 2.8718 ms   time_memory = 7.0000 ms
     arithmetic intensity = 64.00 FLOPs/byte  (hardware ridge point = 156.00 FLOPs/byte)

Prefill/decode, batch=512 (7B, fp16, weights reused across batch)
  -> compute-bound
     time_compute = 22.9744 ms   time_memory = 7.0000 ms
     arithmetic intensity = 512.00 FLOPs/byte  (hardware ridge point = 156.00 FLOPs/byte)



In [ ]:
# Batch = 1: Memory-bound → most time is spent loading model weights.
# Batch = 64–512: More data is processed together, so weight loading is reused → eventually becomes compute-bound.
# Memory-bound: Reduce data movement → quantization, KV-cache optimization, better batching.
# Compute-bound: Reduce or speed up calculations (FLOPs) → optimized kernels, lower precision, sparsity.